In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
import warnings
warnings.filterwarnings("ignore")

c:\Users\admin\Desktop\LearningPython\env\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [3]:
df = pd.read_csv("fraud_detection_dataset.csv")

In [4]:
df.head()

,transaction_id,customer_id,timestamp,transaction_amount,avg_transaction_amount_30d,amount_deviation,txn_velocity,velocity_change,location_change_km,merchant_category_risk,hour_of_day
0,T0001_001,C0001,2025-11-01 08:00:00,1868.30,1626,1.15,0,0.0,0.0,0.10,19
1,T1792_001,C1792,2025-11-01 08:00:00,1639.10,926,1.77,0,0.0,0.0,0.15,12
2,T0044_001,C0044,2025-11-01 08:00:00,2372.65,2557,0.93,0,0.0,0.0,0.40,22
3,T1244_001,C1244,2025-11-01 08:00:00,2523.01,2170,1.16,0,0.0,0.0,0.15,8
4,T1001_001,C1001,2025-11-01 08:00:00,2014.75,1684,1.20,0,0.0,0.0,0.10,16


In [5]:
df.tail()

,transaction_id,customer_id,timestamp,transaction_amount,avg_transaction_amount_30d,amount_deviation,txn_velocity,velocity_change,location_change_km,merchant_category_risk,hour_of_day
31528,T1378_020,C1378,2025-12-04 19:51:00,2066.75,2425,0.85,1,0.5,0.0,0.20,10
31529,T1850_016,C1850,2025-12-04 20:40:00,57711.00,1458,39.58,0,0.0,0.0,0.85,1
31530,T1208_019,C1208,2025-12-04 21:33:00,1944.51,1458,1.33,0,0.0,0.0,0.30,10
31531,T0664_016,C0664,2025-12-04 22:02:00,2233.31,2637,0.85,0,0.0,0.0,0.20,8
31532,T1850_017,C1850,2025-12-04 23:50:00,41254.26,1458,28.30,0,0.0,5837.1,0.85,1


In [6]:
high_velocity_df = df[df["txn_velocity"] > 1]

In [7]:
high_velocity_df

,transaction_id,customer_id,timestamp,transaction_amount,avg_transaction_amount_30d,amount_deviation,txn_velocity,velocity_change,location_change_km,merchant_category_risk,hour_of_day


In [8]:
df = df.sort_values(["customer_id", "timestamp"])

In [9]:
df["amount_deviation"] = df["transaction_amount"] / (df["avg_transaction_amount_30d"] + 1)
df["velocity_change"] = df["txn_velocity"] / (df["txn_velocity"].rolling(10).mean() + 1)

In [10]:
df.fillna(0, inplace=True)

In [11]:
num_features = [
    "transaction_amount",
    "avg_transaction_amount_30d",
    "txn_velocity",
    "amount_deviation",
    "velocity_change",
    "location_change_km",
    "merchant_category_risk",
    "hour_of_day"
]

In [12]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[num_features])


In [13]:
iso = IsolationForest(
    n_estimators=300,
    contamination=0.03,
    random_state=42
)
iso.fit(X_scaled)


,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",300
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.03
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary ` for more details.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",42
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary `... versionadded:: 0.21",False


In [14]:
df["iso_score"] = -iso.decision_function(X_scaled)  # higher = more anomaly
df["iso_flag"] = iso.predict(X_scaled).astype(int)
df["iso_flag"] = df["iso_flag"].apply(lambda x: 1 if x == -1 else 0)


In [15]:
SEQ_LEN = 10
sequence_features = ["transaction_amount", "txn_velocity", "amount_deviation"]

In [16]:
def create_sequences(values, seq_len):
    seq = []
    for i in range(len(values) - seq_len):
        seq.append(values[i:i+seq_len])
    return np.array(seq)

In [17]:
sequences = []
for cust_id, group in df.groupby("customer_id"):
    arr = group[sequence_features].values
    seq_data = create_sequences(arr, SEQ_LEN)
    if len(seq_data) > 0:
        sequences.append(seq_data)

In [18]:
X_seq = np.vstack(sequences)

In [19]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense


In [20]:
lstm_model = Sequential([
    LSTM(64, input_shape=(SEQ_LEN, len(sequence_features))),
    Dense(32, activation="relu"),
    Dense(1)
])

In [21]:
lstm_model.compile(optimizer="adam", loss="mse")

In [22]:
lstm_model.fit(X_seq, np.mean(X_seq, axis=(1,2)), epochs=8, batch_size=64, verbose=0)

In [23]:
def get_lstm_score(customer_df):
    values = customer_df[sequence_features].values
    if len(values) < SEQ_LEN:
        return 0
    seq = values[-SEQ_LEN:]
    seq = seq.reshape(1, SEQ_LEN, len(sequence_features))
    pred = lstm_model.predict(seq, verbose=0)[0][0]
    actual = values[-1].mean()
    err = abs(pred - actual)
    return err

In [24]:
df["lstm_score"] = df.groupby("customer_id").apply(get_lstm_score).reset_index(0, drop=True)
df["lstm_score"].fillna(0, inplace=True)

KeyboardInterrupt: 

In [ ]:
def rule_engine(row):
    score = 0
    
    # Rule 1: high-risk merchant
    if row["merchant_category_risk"] > 0.7:
        score += 10

    # Rule 2: unusual time (1 AM – 5 AM)
    if row["hour_of_day"] >= 1 and row["hour_of_day"] <= 5:
        score += 5

    # Rule 3: large amount threshold
    if row["transaction_amount"] > row["avg_transaction_amount_30d"] * 4:
        score += 15

    # Rule 4: sudden geolocation jump
    if row["location_change_km"] > 300:
        score += 10

    return score

In [ ]:
df["rule_score"] = df.apply(rule_engine, axis=1)

In [ ]:
def final_risk(row):
    s = 0
    s += min(30, row["lstm_score"] * 10)           # LSTM anomaly
    s += min(25, row["iso_score"] * 20)            # Isolation forest anomaly
    s += min(20, row["amount_deviation"] * 4)      # Amount risk
    s += min(15, row["velocity_change"] * 3)       # Velocity risk
    s += min(10, row["rule_score"])                # Rule engine

    return min(100, s)

In [ ]:
df["final_risk_score"] = df.apply(final_risk, axis=1)

In [ ]:

df["risk_label"] = df["final_risk_score"].apply(
    lambda x: "CRITICAL" if x >= 85 else ("HIGH" if x >= 70 else ("MEDIUM" if x >= 40 else "LOW"))
)

In [ ]:
def rule_override_for_input(amt, avg30, vel, loc, merch, hour):
    amt_dev = amt / (avg30 + 1)
    vel_dev = vel / (vel + 1)

    score = 0

    # Transaction deviation very high → strong fraud indicator
    if amt_dev > 20:
        score += 2

    # Velocity increase
    if vel > 9:
        score += 1

    # High-risk merchant
    if merch > 0.7:
        score += 2

    # Odd hours (late night)
    if hour >= 22 or hour <= 4:
        score += 1

    if score >= 5:
        return "CRITICAL"
    elif score >= 3:
        return "HIGH"
    else:
        return None  # let ML model decide


In [ ]:
amt = 20000
avg30 = 12000
vel = 0
loc = 0
merch = 0.25
hour = 12
amt_dev = amt / (avg30 + 1)
vel_dev = vel / (vel + 1)
override_result = rule_override_for_input(amt, avg30, vel, loc, merch, hour)

In [ ]:
 if override_result is not None:
        print("\n------ RESULT (RULE OVERRIDE) ------")
        print("Risk Label:", override_result)
        print("Reason: Rule-based override triggered")
        print("------------------------------------\n")

In [ ]:
features = np.array([[amt, avg30, vel, amt_dev, vel_dev, loc, merch, hour]])
X_scaled = scaler.transform(features)
iso_s = -iso.decision_function(X_scaled)[0]

In [ ]:
lstm_s = abs(amt_dev - 1)

In [ ]:
temp = {
        "transaction_amount": amt,
        "avg_transaction_amount_30d": avg30,
        "txn_velocity": vel,
        "location_change_km": loc,
        "merchant_category_risk": merch,
        "hour_of_day": hour
    }
rule_s = rule_engine(pd.Series(temp))


In [ ]:
    final = (
        min(30, lstm_s * 10)
        + min(25, iso_s * 20)
        + min(20, amt_dev * 4)
        + min(15, vel_dev * 3)
        + min(10, rule_s)
    )

    final = min(100, final)

    if final >= 85:
        label = "CRITICAL"
    elif final >= 70:
        label = "HIGH"
    elif final >= 40:
        label = "MEDIUM"
    else:
        label = "LOW"


In [ ]:
print("\n------ RESULT ------")
print("Final Risk Score:", final)
print("Risk Label:", label)
print("--------------------\n")


------ RESULT ------
Final Risk Score: 59.650125025650375
Risk Label: MEDIUM
--------------------

